# Agregações de Vendas (camada Gold)

Este notebook cria as tabelas agregadas da camada **Gold** a partir do fato de vendas (`fact_sales`) e das tabelas de dimensões (região, canal, data e marca). Ele consolida as vendas por diferentes perspectivas analíticas:

- **4.1 — Região × Trade Group** (`agg_sales_region_trade_group`): total de volume de vendas por região e grupo de trade, usando a descrição de canal.
- **4.2 — Marca × Mês** (`agg_sales_brand_month`): total de volume de vendas por marca e mês.
- **4.3 — Região × Marca** (`agg_sales_region_brand`): total de volume de vendas por região e marca.

O catálogo de destino é definido pelo widget `CATALOG`; as tabelas são criadas/substituídas como tabelas Delta no schema `<CATALOG>.gold`.


In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("CATALOG", "")

CATALOG = dbutils.widgets.get("CATALOG")
G = f"{CATALOG}.gold"



In [0]:
# 4.1: região x trade group (TRADE_GROUP_DESC vem do arquivo de canal)
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.agg_sales_region_trade_group
    USING DELTA
    COMMENT 'Vendas por região e trade group (pergunta 4.1).' AS
    SELECT r.region_name, c.trade_group_desc,
        SUM(f.sales_volume) AS total_sales_volume,
        COUNT(*)            AS n_rows
    FROM {G}.fact_sales f
    JOIN {G}.dim_region  r ON f.region_key  = r.region_key
    JOIN {G}.dim_channel c ON f.channel_key = c.channel_key
    GROUP BY r.region_name, c.trade_group_desc
""")

In [0]:
# 4.2: marca x mês
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.agg_sales_brand_month
    USING DELTA
    COMMENT 'Vendas por marca e mês (pergunta 4.2).' AS
    SELECT d.year_month, b.brand_nm,
        SUM(f.sales_volume) AS total_sales_volume,
        COUNT(*)            AS n_rows
    FROM {G}.fact_sales f
    JOIN {G}.dim_date  d ON f.date_key  = d.date_key
    JOIN {G}.dim_brand b ON f.brand_key = b.brand_key
    GROUP BY d.year_month, b.brand_nm
""")

In [0]:
# 4.3: região x marca
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.agg_sales_region_brand
    USING DELTA
    COMMENT 'Vendas por região e marca (pergunta 4.3).' AS
    SELECT r.region_name, b.brand_nm,
        SUM(f.sales_volume) AS total_sales_volume,
        COUNT(*)            AS n_rows
    FROM {G}.fact_sales f
    JOIN {G}.dim_region r ON f.region_key = r.region_key
    JOIN {G}.dim_brand  b ON f.brand_key  = b.brand_key
    GROUP BY r.region_name, b.brand_nm
""")